# E-Commerce Return Intelligence
## AI-Powered Product Return Prediction & Big Data Analytics
**Final-Year Academic Project | BTech CSE Data Science**

This notebook provides end-to-end data exploration, Big Data processing via PySpark MLlib, and Scikit-Learn machine learning pipeline development for e-commerce return prediction.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Dataset Loading and Initial Exploration
We load `Ecommerce_2000.csv` containing 2,000 raw transaction records.

In [2]:
df_raw = pd.read_csv("../dataset/Ecommerce_2000.csv")
print("Raw shape:", df_raw.shape)
print("Duplicate rows:", df_raw.duplicated().sum())
df_clean = df_raw.drop_duplicates().copy()
print("Clean shape:", df_clean.shape)
print("Target Distribution:")
print(df_clean["Returned"].value_counts(normalize=True))

## 2. Big Data Processing with PySpark MLlib (Tasks 121 - 125)
Here we instantiate SparkSession and execute distributed data transformation and modeling.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("EcommerceReturnAnalysis") \
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark Version:", spark.version)

spark_df = spark.read.csv("../dataset/Ecommerce_2000.csv", header=True, inferSchema=True)
spark_df_dedup = spark_df.dropDuplicates()
print(f"Spark records: {spark_df_dedup.count()}")

## 3. Scikit-Learn Pipeline & Feature Engineering
Preparing the deployable pipeline with 70% Train, 15% Validation, and 15% Test split.

In [4]:
sys.path.append('..')
from src.data_preprocessing import prepare_training_data
from src.feature_engineering import extract_features, build_preprocessor
from src.train_models import train_and_evaluate

results = train_and_evaluate(
    data_path="../dataset/Ecommerce_2000.csv",
    output_dir="../models",
    predictions_dir="../output"
)
print("Selected Model:", results["selected_model"])
print("Optimal Threshold:", results["threshold"])

## 4. Model Evaluation & Conclusion
Evaluating final metrics on the untouched test set and analyzing key drivers of returns.

In [5]:
test_metrics = results["test_metrics"][results["selected_model"]]
print("=== Final Untouched Test Set Metrics ===")
for k, v in test_metrics.items():
    if k != "classification_report":
        print(f"{k:18s}: {v}")